#### This notebook calculates mean areal temperature (MAT) for the 40-year NWM v3.0 retrospective for USGS high-res basin polygons (CONUS only)
- Requires the weights (fractional pixel coverage) to be pre-calculated (see `05_calculate_nwm30_pixel_weights.ipynb`)
- Reads NWM v3.0 forcing (T2D) netcdf files from s3 using Sedona: https://noaa-nwm-retrospective-3-0-pds.s3.amazonaws.com/index.html#CONUS/
- Writes output to local TEEHR Evaluation (warehouse)
- NOTE:
    -  The `T2D` variable contains a `scale factor` and `offset` parameters in the netcdf file that was not applied by Sedona during `RS_ReadFromNETCDF`
    -  This needed to be applied manually, need a better method to check if these exist
    -  In the future we should consider writing with `insert` rather than `append` to save time, since `append` uses `NERGE INTO`. And check for duplicates after processing.

In [ ]:
import os
from pathlib import Path
import logging
import shutil
import time
import gc
import glob
import re

from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType
import xarray as xr
import fsspec
import rioxarray
import rasterio
import geopandas as gpd
import numpy as np
import pandas as pd

import teehr
from teehr.evaluation.spark_session_utils import create_spark_session


logger = logging.getLogger(__name__)

teehr.__version__

Configure the logger

In [ ]:
logger.setLevel(logging.DEBUG)
file_handler = logging.FileHandler('sedona_mat_processor.log')
file_handler.setLevel(logging.DEBUG)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

In [ ]:
# spark.stop()

In [ ]:
%%time
# ~2 pods/node
NUM_EXECUTORS = 50
NUM_CORES = 7
EXECUTOR_MEMORY = "50g"

NUM_SHUFFLE_PARTITIONS = NUM_EXECUTORS * NUM_CORES * 2

# spark = create_spark_session()
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=NUM_EXECUTORS,
    executor_memory=EXECUTOR_MEMORY,
    executor_cores=NUM_CORES,
    aws_region="us-east-1",
    update_configs={
        "spark.hadoop.fs.s3a.aws.credentials.provider":
        "org.apache.hadoop.fs.s3a.AnonymousAWSCredentialsProvider",
        "spark.sql.shuffle.partitions": f"{NUM_SHUFFLE_PARTITIONS}",
    }    
)

#### Set up the local evaluation. We'll write the MAT here

In [ ]:
dir_path = "/data/playground/slamont/teehr/warehouse/sedona/mat_processing/mat_warehouse"

# USE EXISTING:
ev = teehr.LocalReadWriteEvaluation(
    spark=spark,
    dir_path=dir_path,
    create_dir=False
)

These steps only need to be performed one time:

In [ ]:
ev.locations.load_spatial(in_path="/data/playground/slamont/teehr/warehouse/sedona/usgs_basin_geometry/usgsbasin_geometry_highres.conus.nwmCRS.parquet")

Get the weights table from the TEEHR-Cloud warehouse and write to local

In [ ]:
weights_table_sdf = ev.table("grid_pixel_coverage_weights", catalog_name="iceberg", namespace_name="teehr").to_sdf()

ev.write.to_warehouse(
    table_name="grid_pixel_coverage_weights",
    source_data=weights_table_sdf,
    write_mode="create_or_replace"
)

In [ ]:
ev.table("variables", catalog_name="iceberg", namespace_name="teehr").to_sdf().show(truncate=False)

Add the `variable_name` and `unit_name` for temperature

In [ ]:
tvar = teehr.Variable(long_name="Hourly Mean Temperature", name="temperature_hourly_mean")
ev.variables.add(tvar)

tunit = teehr.Unit(name="K", long_name="Kelvins")
ev.units.add(tunit)

Apply the latest partitioning migration to potentially help with writes

In [ ]:
from teehr.utilities.apply_migrations import evolve_catalog_schema

migrations_dir = "/home/jovyan/git/teehr-hub/warehouse/migrations"

evolve_catalog_schema(
    spark=spark,
    migrations_dir_path=migrations_dir,
    target_catalog_name="local",
    target_namespace_name="teehr"
)

#### Begin the MAT Workflow

In [ ]:
DATE_PATTERN = r"(\d{12})"
CONFIGURATION_NAME = "nwm30_retrospective"
UNIT_NAME = "K"
TEEHR_VARIABLE_NAME = "temperature_hourly_mean"

NWM_VARIABLE_NAME = "T2D"

# Create filepath generator
BASE_S3_PATH = "s3a://noaa-nwm-retrospective-3-0-pds/CONUS/netcdf/FORCING/"

START_DATE = "2022-10-04 19:00:00"     # First file: 1979-02-01 00:00
END_DATE = "2023-01-31 23:00"       # Last file: 2023-01-31 23:00

CHUNK_SIZE = 220  # Number of timesteps (hours) processed at once (1 year test used 200, but can go higher)  INCREASE for more efficiency/less IO but more memory

In [ ]:
start_date = pd.Timestamp(START_DATE)
end_date = pd.Timestamp(END_DATE)
dt_rng = pd.date_range(start=START_DATE, end=END_DATE, freq="h")

NUM_SPLITS = int(len(dt_rng) / CHUNK_SIZE)
full_filepaths = [f"{BASE_S3_PATH}{dt.year}/{dt.year}{dt.month:02d}{dt.day:02d}{dt.hour:02d}00.LDASIN_DOMAIN1" for dt in dt_rng]

split_full_filepaths = np.array_split(full_filepaths, NUM_SPLITS)

len(split_full_filepaths)

In [ ]:
%%time
LOCATION_ID_PREFIX = "usgsbasin"  # The locations to use (currently only usgs basins are in the weights table)

# Create a view of fractional coverage table
spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW fractions_view AS
    SELECT fraction_covered, location_id, position_index
    FROM local.teehr.grid_pixel_coverage_weights
    WHERE location_id LIKE '{LOCATION_ID_PREFIX}-%'
""")
spark.sql("CACHE TABLE fractions_view")

In [ ]:
%%time
table_name = "primary_timeseries"

cntr = 0
for i, split_filepaths_chunk in enumerate(split_full_filepaths):

    t0 = time.time()

    filepaths = [str(fp) for fp in split_filepaths_chunk]

    nc_sdf = spark.read.format("binaryFile").load(filepaths).selectExpr(f"RS_FromNetCDF(content, '{NWM_VARIABLE_NAME}', 'x', 'y') as raster", "path as filepath") 
    nc_sdf = nc_sdf.withColumn("value_time", F.regexp_extract(nc_sdf["filepath"], DATE_PATTERN, 1))  # partition by time?
    
    # Explode the raster values
    raster_exp_sdf = nc_sdf.selectExpr(
        "posexplode(RS_BandAsArray(raster, 1))",
        "value_time",
    ).selectExpr(
        "value_time as value_time",
        "col as value",
        "CAST(pos as BIGINT) as position_index"
    )
    raster_exp_sdf.createOrReplaceTempView("raster_values")
    
    # Calculate MAP
    map_results = spark.sql(f"""
        SELECT /*+ BROADCAST(w) */
            w.location_id,
            to_timestamp(r.value_time, 'yyyyMMddHHmm') AS value_time,
            SUM(r.value * w.fraction_covered) / SUM(w.fraction_covered) AS value,
            CAST(NULL AS TIMESTAMP) AS reference_time,
            '{UNIT_NAME}' AS unit_name,
            '{TEEHR_VARIABLE_NAME}' AS variable_name,
            '{CONFIGURATION_NAME}' AS configuration_name
        FROM 
            raster_values AS r
        JOIN 
             fractions_view AS w ON r.position_index = w.position_index
        GROUP BY 
            w.location_id, r.value_time;
    """)
    
    # Write to table
    ev.write.to_warehouse(
        source_data=map_results,
        table_name=table_name,  # Note. 1-year run stored in "temp_secondary_timeseries" table
        write_mode="append"
    )

    spark.catalog.dropTempView("raster_values")

    del raster_exp_sdf
    gc.collect()
    logger.info(f"Processed chunk {i}/{len(split_full_filepaths)} in {(time.time() - t0) / 60:.2f} mins")

#### Check out the start/end times, total timesteps, etc.

In [ ]:
sdf = ev.primary_timeseries.to_sdf()

In [ ]:
sdf.select(F.min("value_time")).show(), sdf.select(F.max("value_time")).show()

#### NOTE: scale and offset values must be applied. See post-processing